In [2]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from huggingface_hub import login

In [6]:
path = Path.cwd()
data_dir = path.parent / "data"
preprocess_data = data_dir / "preprocess"

In [2]:
login(token=your_token)

In [4]:
from transformers import AutoTokenizer , AutoModel

model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name,
    device_map="cuda",          # FORCES it completely into VRAM (no CPU split)
    torch_dtype=torch.float16,  # Cuts RAM/VRAM usage in half
    low_cpu_mem_usage=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PosixPath('/home/rohit_vastani/Desktop/New Project/data/raw')

In [7]:
data = pd.read_csv(preprocess_data / "indic-language.csv")
data.head()

,input,label
0,कनाडा अमेरिका और यूरोपीय संघ का अनुसरण करते हु...,Hindi
1,विदेशों में मूलधातुओं की कीमतों में कमजोरी के ...,Hindi
2,डेविड वॉर्नर पर क्रिकेट ऑस्ट्रेलिया ने 1 साल क...,Hindi
3,"अगर आपके पास फटे-पुराने नोट हैं, जिन्हें दुक...",Hindi
4,नोवेल लवासा ने देर रात बयान जारी कर कहा कि उन्...,Hindi


In [6]:
data["label"].value_counts()

,count
label,
Hindi,500
Gujarati,500
Punjabi,500
Marathi,500
Tamil,500
Telugu,500
Malayalam,500
Kannada,500


In [9]:
BATCH_SIZE = 32
X = []
text = data["input"].to_list()

for i in tqdm(range(0, len(text), BATCH_SIZE)):
  #load the batch
  batch_text = text[i : i + BATCH_SIZE]

  # input
  inputs = tokenizer(
      batch_text,
      padding=True,
      truncation=True,
      max_length=128,
      return_tensors="pt"
  ).to("cuda")

  # generate embeddings
  with torch.no_grad():
    outputs = model(**inputs)

  batch_embeddings = outputs.last_hidden_state.mean(dim=1).float().cpu().numpy()
  X.extend(batch_embeddings)

X = np.array(X)
print("Final shape: ", X.shape)

100%|██████████| 125/125 [00:12<00:00,  9.89it/s]

Final shape:  (4000, 768)


In [8]:
embeddings_data = data_dir / "embeddings"

In [ ]:
np.save(embeddings_data / "indic_bert_v1.npy", X)